In [ ]:
# Resolve paths consistently from either the repository root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src').is_dir() and (p / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Run this notebook from within the cloned repository.')
os.chdir(PROJECT_ROOT / 'notebooks')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Parsing crime data from FBI

In [ ]:
#sets the notebook width
from IPython.display import display, HTML
display(HTML("<style>.container { width:85% !important; }</style>"))
%load_ext autoreload
%autoreload 2

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Standard Libraries
import sys
import os
from pathlib import Path
import glob
import math

In [ ]:
# Third party libraries
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats


In [ ]:
# figures
import matplotlib.mlab as ml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import matplotlib.ticker as plticker
import seaborn as sns


# Load data

In [ ]:
state_dict_df=pd.read_csv('../data/state_abbreviations.csv')
state_dict_df.head()
state_dict_df

In [ ]:
statemap={50: 'AK',
 1: 'AL',
 3: 'AR',
 54: 'AS',
 2: 'AZ',
 4: 'CA',
 5: 'CO',
 6: 'CT',
 52: 'CZ',
 8: 'DC',
 7: 'DE',
 9: 'FL',
 10: 'GA',
 55: 'GM',
 51: 'HI',
 14: 'IA',
 11: 'ID',
 12: 'IL',
 13: 'IN',
 15: 'KS',
 16: 'KY',
 17: 'LA',
 20: 'MA',
 19: 'MD',
 18: 'ME',
 21: 'MI',
 22: 'MN',
 24: 'MO',
 23: 'MS',
 25: 'MT',
 26: 'NE',# Originally this is NB
 32: 'NC',
 33: 'ND',         
 28: 'NH',
 29: 'NJ',
 30: 'NM',
 27: 'NV',
 31: 'NY',
 34: 'OH',
 35: 'OK',
 36: 'OR',
 37: 'PA',
 53: 'PR',
 38: 'RI',
 39: 'SC',
 40: 'SD',
 41: 'TN',
 42: 'TX',
 43: 'UT',
 62: 'VI',
 45: 'VA',
 44: 'VT',
 46: 'WA',
 48: 'WI',
 47: 'WV',
 49: 'WY'};

In [ ]:
def create_path(year):
    filepath=Path('../data/raw_crime_data/{}.dat'.format(year))
    return filepath

def read_lines(filepath):
    file = open(filepath)
    contents=file.read()
    lines=contents.split('\n')
    return lines

In [ ]:
year=2000
filepath=Path('../data/raw_crime_data/{}.dat'.format(year))
file = open(filepath)
contents=file.read()

# Select and clean data

In [ ]:
def get_cards_dict(monthlabel='january'):
    mydict=dict()
    keys=['Murder',
          'Manslaughter',
          'Rape Total',
          'Rape by Force',
          'Attempted Rape',
          'Robbery Total',
          'Robbery With a Gun',
          'Robbery With a Knife',
          'Robbery - Other Weapon',
          'Strong-Arm Robbery',
          'Assault Total',
          'Assault With a Gun',
          'Assault With a Knife',
          'Assault - Other Weapon',
          'Assault With Hands, Feet, etc',
          'Simple Assault',
          'Burglary Total',
          'Burglary - Forcible Entry',
          'Burglary - Not Forcible Entry',
          'Attempted Burglary',
          'Larceny Total',
          'Motor Vehicle Theft Total',
          'Auto Theft',
          'Truck, Bus Theft',
          'Other Vehicle Theft',
          'Grand Total of All Fields',
          'Lacerny Under $50']
    for i,element in enumerate(keys):
        keys[i]=monthlabel+'_'+element
    description=['Number of unfounded offenses',
             'Number of actual offenses',
             'Total offeses cleared by arrest',
             'Number of clearances under 18' 
            ]
    #starts=[323,463,603,743]
    monthdict={ 'january':306,
                'february':896,
                'march':1486,
                'april':2076,
                'may':2666,
                'june':3256,
                'july':3846,
                'august':4436,
                'september':5026,
                'october':5616,
                'november':6206,
                'december':6796
              }
    month=monthdict[monthlabel]+17
    starts=[month,month+140,month+280,month+420]
    cards=dict()
    for j,start in enumerate(starts):
        mydict=dict()
        begin=start
        end=start+4
        for i in range(0,27):
            mydict[keys[i]]=(begin,end)
            begin=begin+5
            end=begin+4
        cards[j]=mydict
    return cards,description

In [ ]:
positions_dict={'State':(2,3),
                'ORI':(4,10),
                'Group':(11,12),
                'Division':(13,13),
                'Year':(14,15),
                'Sequence Number':(16,20),
                'Juvenile Age':(21,22),
                'Core City Indication':(23,23),
                'Covered By':(24,30),
                'Covered By Group':(31,31),
                'Last Update':(32,37),
                'Field Office':(38,41),
                'Number of Months Reported':(42,43),
                'Agency Count':(44,44),
                'Population':(45,53),
                'County':(54,56),
                'MSA':(57,59),
                'Group I':(60,74),
                'Group II':(75,89),
                'Agency Name':(121,144),
                'Agency State Name':(145,150),
                'First Line of Mailing Address':(151,180),
                'Second Line of Mailing Address':(181,210),
                'Third Line of Mailing Address':(211,240),
                'Fourth Line of Mailing Address':(241,270),
                'Zip Code':(271,275)
               }

In [ ]:
months=['january', 'february', 'march', 'april', 'may', 'june',
        'july', 'august', 'september', 'october', 'november', 'december']

In [ ]:
filepath = create_path(1960)
print(filepath)
lines=read_lines(filepath)

In [ ]:
def to_dataframe(year,cards):
    filepath = create_path(year)
    print(filepath)
    lines=read_lines(filepath)
    columns=list( positions_dict.keys())+list(cards.keys())

    df=pd.DataFrame(columns=columns).T
    for k,line in enumerate(lines):
        values=[]
        for column in positions_dict.keys():
            start,end=positions_dict[column]
            value=line[start-1:end].strip()
            values.append(value)
            #print(column,value)
        for column in cards.keys():
            start,end= cards[column]
            value=line[start-1:end].strip()
            #print(column,value)
            values.append(value)
        df[k]=values
    df=df.T
    return df

In [ ]:
cards_dict={}
for month in months:
    cards,description=get_cards_dict(monthlabel=month)
    cards_dict.update(cards[1])

In [ ]:
def base_cleaner(x):
    try:
        return int(x)
    except:
        return None

# Save data

In [ ]:
for year in range(1960,2015):
    print('Convert to dataframe')
    df=to_dataframe(year,cards_dict)
    print('Convert to integers')
    for column in  cards_dict.keys():
        df[column]= df[column].apply(base_cleaner)
    print('Saving')
    df.to_csv('../data/clean_crime_data/{}.csv'.format(year))
    print('Done')
    print()